# Where is Waldo? — Exploration Notebook

Interactive exploration of every backend module:

| # | Section | Modules covered |
|---|---------|----------------|
| 1 | Environment setup | — |
| 2 | Scene generation | `generate_scene`, `backgrounds`, `characters` |
| 3 | Difficulty comparison | `DIFFICULTY_TO_COUNT`, `generate_scene` |
| 4 | Sprite inspection | `waldo_sprite`, `random_character_sprite` |
| 5 | Collision utilities | `overlap_area`, `area`, `can_place_bbox` |
| 6 | BBox utilities | `bbox_to_yolo`, `yolo_to_bbox`, `point_in_bbox`, `iou` |
| 7 | Image utilities | `pil_to_numpy_bgr`, `numpy_bgr_to_pil` |
| 8 | Dataset builder | `generate_dataset` |
| 9 | YOLO inference | `detect_waldo` |
| 10 | Full game simulation | all modules |


## 1 — Environment Setup

In [1]:
import sys
from pathlib import Path

# Add project root to path so backend imports work from the notebook
PROJECT_ROOT = Path().resolve().parents[1]  # frontend/notebooks -> project root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")

Project root: E:\projects\folder-where-is-waldo-yolo
Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [2]:
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

# Inline figures, higher DPI
%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "#272b30", "text.color": "#aab3bc"})

print("Imports OK")

Imports OK


## 2 — Scene Generation

`generate_scene(num_people, difficulty, image_size, seed)` returns `(PIL.Image, bbox)`  
where `bbox = (x1, y1, x2, y2)` in pixels.

In [3]:
from backend.scene_generation.generate_scene import generate_scene

# Generate a reproducible medium scene
scene, waldo_bbox = generate_scene(difficulty="medium", seed=42)

print(f"Scene size : {scene.size}  (W x H)")
print(f"Waldo bbox : {waldo_bbox}  (x1, y1, x2, y2)")
print(f"Waldo size : {waldo_bbox[2]-waldo_bbox[0]} x {waldo_bbox[3]-waldo_bbox[1]} px")

ModuleNotFoundError: No module named 'backend'

In [ ]:
def show_scene(img: Image.Image, bbox: tuple, title: str = "Scene", reveal: bool = False):
    """Display a scene with optional ground-truth bbox overlay."""
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(img)
    ax.set_title(title, color="#aab3bc", fontsize=13, pad=10)
    ax.axis("off")

    if reveal:
        x1, y1, x2, y2 = bbox
        rect = mpatches.FancyBboxPatch(
            (x1, y1), x2 - x1, y2 - y1,
            boxstyle="round,pad=2",
            linewidth=3, edgecolor="#5cb85c", facecolor="none",
        )
        ax.add_patch(rect)
        ax.annotate(
            "Waldo!", xy=(x1, y1), xytext=(x1, max(0, y1 - 12)),
            color="#5cb85c", fontsize=9, fontweight="bold",
            bbox=dict(boxstyle="round", fc="black", alpha=0.6),
        )

    plt.tight_layout()
    plt.show()


# Show scene with hidden Waldo (as the player would see it)
show_scene(scene, waldo_bbox, title="Medium scene — can you find Waldo?")

In [ ]:
# Reveal ground truth
show_scene(scene, waldo_bbox, title="Ground truth revealed", reveal=True)

## 3 — Difficulty Comparison

Side-by-side view of all three difficulty levels.

In [ ]:
from backend.utils.config import DIFFICULTY_TO_COUNT

print("Character counts per difficulty:")
for level, count in DIFFICULTY_TO_COUNT.items():
    print(f"  {level:8s} → {count} characters")

In [ ]:
SEED = 7
difficulties = ["easy", "medium", "hard"]
scenes = {d: generate_scene(difficulty=d, seed=SEED) for d in difficulties}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, diff in zip(axes, difficulties):
    img, bbox = scenes[diff]
    ax.imshow(img)
    # Draw ground truth
    x1, y1, x2, y2 = bbox
    rect = mpatches.FancyBboxPatch(
        (x1, y1), x2 - x1, y2 - y1,
        boxstyle="round,pad=2",
        linewidth=2.5, edgecolor="#5bc0de", facecolor="none",
    )
    ax.add_patch(rect)
    count = DIFFICULTY_TO_COUNT[diff]
    ax.set_title(f"{diff.capitalize()}  ({count} chars)", color="#aab3bc", fontsize=12)
    ax.axis("off")

fig.suptitle("Difficulty comparison — Waldo highlighted in blue",
             color="#aab3bc", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4 — Sprite Inspection

Explore `waldo_sprite()` and `random_character_sprite()` at different sizes.

In [ ]:
from backend.scene_generation.characters import waldo_sprite, random_character_sprite

rng = random.Random(0)

# Generate Waldo at multiple sizes
waldo_sizes = [(30, 55), (50, 85), (70, 115), (90, 145)]
waldos = [waldo_sprite(rng, size) for size in waldo_sizes]

# Generate random characters
char_sizes = [(30, 55), (50, 85), (70, 115), (90, 145)]
chars = [random_character_sprite(rng, size) for size in char_sizes]

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, (ax, sprite, size) in enumerate(zip(axes[0], waldos, waldo_sizes)):
    ax.imshow(sprite)
    ax.set_title(f"Waldo {size[0]}×{size[1]}", color="#5bc0de", fontsize=9)
    ax.axis("off")
for i, (ax, sprite, size) in enumerate(zip(axes[1], chars, char_sizes)):
    ax.imshow(sprite)
    ax.set_title(f"Char {size[0]}×{size[1]}", color="#aab3bc", fontsize=9)
    ax.axis("off")

axes[0][0].set_ylabel("Waldo", color="#5cb85c", fontsize=10)
axes[1][0].set_ylabel("Random\ncharacter", color="#aab3bc", fontsize=10)
fig.suptitle("Synthetic sprites at different sizes", color="#aab3bc", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Grid of 12 random characters to show variety
rng2 = random.Random(99)
size = (60, 100)
grid = [random_character_sprite(rng2, size) for _ in range(12)]

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, sprite in zip(axes.flat, grid):
    ax.imshow(sprite)
    ax.axis("off")

fig.suptitle("Character variety — synthetic sprites", color="#aab3bc", fontsize=12)
plt.tight_layout()
plt.show()

## 5 — Collision Utilities

Explore `overlap_area`, `area`, and `can_place_bbox` from `backend.scene_generation.collision`.

In [ ]:
from backend.scene_generation.collision import overlap_area, area, can_place_bbox

# Example bboxes (x1, y1, x2, y2)
box_a = (10, 10, 60, 80)   # Waldo
box_b = (40, 30, 90, 100)  # overlapping character
box_c = (200, 200, 250, 280)  # far away, no overlap

print(f"area(box_a)             = {area(box_a)} px²")
print(f"area(box_b)             = {area(box_b)} px²")
print(f"overlap_area(a, b)      = {overlap_area(box_a, box_b)} px²")
print(f"overlap_area(a, c)      = {overlap_area(box_a, box_c)} px²")

placed = [box_a]
print(f"\ncan_place(b, ratio=0.10) = {can_place_bbox(box_b, placed, max_overlap_ratio=0.10)}")
print(f"can_place(b, ratio=0.40) = {can_place_bbox(box_b, placed, max_overlap_ratio=0.40)}")
print(f"can_place(c, ratio=0.10) = {can_place_bbox(box_c, placed, max_overlap_ratio=0.10)}")

In [ ]:
# Visualise overlap
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

def draw_box(ax, box, color, label):
    x1, y1, x2, y2 = box
    rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                                linewidth=2, edgecolor=color,
                                facecolor=color, alpha=0.3, label=label)
    ax.add_patch(rect)

for ax, title, boxes in [
    (axes[0], "Overlapping boxes (a, b)", [(box_a, "#5cb85c", "box_a"), (box_b, "#d9534f", "box_b")]),
    (axes[1], "No-overlap pair (a, c)",   [(box_a, "#5cb85c", "box_a"), (box_c, "#5bc0de", "box_c")]),
]:
    for box, color, label in boxes:
        draw_box(ax, box, color, label)
    ax.set_xlim(0, 300)
    ax.set_ylim(0, 300)
    ax.invert_yaxis()
    ax.set_title(title, color="#aab3bc", fontsize=10)
    ax.legend(loc="lower right", fontsize=8)
    ax.set_facecolor("#3a3f44")
    ax.tick_params(colors="#7a8288")

plt.tight_layout()
plt.show()

## 6 — BBox Utilities

Explore `bbox_to_yolo`, `yolo_to_bbox`, `point_in_bbox`, and `iou` from `backend.utils.bbox_utils`.

In [ ]:
from backend.utils.bbox_utils import bbox_to_yolo, yolo_to_bbox, point_in_bbox, iou, clamp_bbox

IMG_W, IMG_H = 640, 640
pixel_bbox = (100, 150, 180, 280)  # x1, y1, x2, y2

# Pixel → YOLO normalised
yolo = bbox_to_yolo(pixel_bbox, IMG_W, IMG_H)
print("Pixel → YOLO:")
print(f"  pixel  : {pixel_bbox}")
print(f"  yolo   : cx={yolo[0]:.4f}, cy={yolo[1]:.4f}, w={yolo[2]:.4f}, h={yolo[3]:.4f}")

# YOLO normalised → pixel (round-trip)
recovered = yolo_to_bbox(*yolo, IMG_W, IMG_H)
print(f"  round-trip pixel: {recovered}")
print(f"  match: {pixel_bbox == recovered}")

# point_in_bbox
print("\npoint_in_bbox:")
for px, py in [(140, 200), (50, 50), (180, 280), (181, 281)]:
    inside = point_in_bbox(px, py, pixel_bbox)
    mark = "✓" if inside else "✗"
    print(f"  ({px:3d},{py:3d})  {mark}  inside={inside}")

# IoU
b1 = (100, 100, 200, 200)
b2 = (150, 150, 250, 250)  # 25 % overlap
b3 = (200, 200, 300, 300)  # touching corner
b4 = (100, 100, 200, 200)  # identical
print("\nIoU:")
print(f"  iou(b1, b2) = {iou(b1, b2):.4f}")
print(f"  iou(b1, b3) = {iou(b1, b3):.4f}")
print(f"  iou(b1, b4) = {iou(b1, b4):.4f}  (identical)")

In [ ]:
# Visualise IoU between two boxes
fig, ax = plt.subplots(figsize=(5, 5))
ax.set_facecolor("#3a3f44")

for box, color, label in [
    (b1, "#5cb85c", f"b1  IoU={iou(b1,b2):.2f}"),
    (b2, "#d9534f", "b2"),
]:
    x1,y1,x2,y2 = box
    rect = mpatches.Rectangle((x1,y1), x2-x1, y2-y1,
                                linewidth=2, edgecolor=color,
                                facecolor=color, alpha=0.35, label=label)
    ax.add_patch(rect)

ax.set_xlim(80, 320)
ax.set_ylim(80, 320)
ax.invert_yaxis()
ax.legend(fontsize=9)
ax.set_title("IoU visualisation", color="#aab3bc")
ax.tick_params(colors="#7a8288")
plt.tight_layout()
plt.show()

## 7 — Image Utilities

Explore `pil_to_numpy_bgr` and `numpy_bgr_to_pil` from `backend.utils.image_utils`.

In [ ]:
from backend.utils.image_utils import pil_to_numpy_bgr, numpy_bgr_to_pil, load_rgba_image

sample_img, sample_bbox = generate_scene(difficulty="easy", seed=1)

# PIL → BGR numpy
bgr_array = pil_to_numpy_bgr(sample_img)
print(f"PIL image size : {sample_img.size}")
print(f"BGR array shape: {bgr_array.shape}  dtype={bgr_array.dtype}")
print(f"BGR array range: [{bgr_array.min()}, {bgr_array.max()}]")

# BGR numpy → PIL (round-trip)
recovered_pil = numpy_bgr_to_pil(bgr_array)
print(f"Round-trip size: {recovered_pil.size}")

In [ ]:
# Visual check: original vs round-trip
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(sample_img)
axes[0].set_title("Original PIL", color="#aab3bc")
axes[0].axis("off")
axes[1].imshow(recovered_pil)
axes[1].set_title("After PIL→BGR→PIL round-trip", color="#aab3bc")
axes[1].axis("off")
plt.tight_layout()
plt.show()

# Pixel difference
diff = np.abs(np.array(sample_img).astype(int) - np.array(recovered_pil).astype(int))
print(f"Max pixel difference (round-trip): {diff.max()}")

## 8 — Dataset Builder

Generate a small synthetic dataset and inspect the output format.

In [ ]:
from backend.vision.dataset.dataset_builder import generate_dataset
from backend.utils.config import get_paths

paths = get_paths()
print("Dataset paths:")
print(f"  images : {paths.images_dir}")
print(f"  labels : {paths.labels_dir}")
print(f"  yaml   : {paths.dataset_yaml}")

In [ ]:
# Generate a mini dataset (10 images) for exploration
# Change n_images to 2000 for real training
N_IMAGES = 10

print(f"Generating {N_IMAGES} synthetic images...")
generate_dataset(n_images=N_IMAGES)
print("Done.")

# Count outputs
images = sorted(paths.images_dir.glob("*.png"))
labels = sorted(paths.labels_dir.glob("*.txt"))
print(f"  {len(images)} images, {len(labels)} labels")

In [ ]:
# Inspect one image + its YOLO label
if images:
    idx = 0
    img_path = images[idx]
    lbl_path = labels[idx]

    img = Image.open(img_path)
    label_text = lbl_path.read_text().strip()
    print(f"Image : {img_path.name}  size={img.size}")
    print(f"Label : {label_text}")

    # Parse YOLO label
    parts = list(map(float, label_text.split()))
    cls_id, cx, cy, bw, bh = parts
    print(f"  class_id={int(cls_id)}, cx={cx:.4f}, cy={cy:.4f}, w={bw:.4f}, h={bh:.4f}")

    # Convert back to pixels for visualisation
    W, H = img.size
    x1 = int((cx - bw/2) * W)
    y1 = int((cy - bh/2) * H)
    x2 = int((cx + bw/2) * W)
    y2 = int((cy + bh/2) * H)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(img)
    rect = mpatches.FancyBboxPatch(
        (x1, y1), x2-x1, y2-y1,
        boxstyle="round,pad=2",
        linewidth=2.5, edgecolor="#5cb85c", facecolor="none",
    )
    ax.add_patch(rect)
    ax.set_title(f"YOLO label overlay — {img_path.name}", color="#aab3bc")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No images found — run generate_dataset() first.")

In [ ]:
# Show a grid of all generated images with labels
n_show = min(len(images), 9)
cols = 3
rows = (n_show + cols - 1) // cols

if n_show > 0:
    fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4))
    axes = axes.flat if hasattr(axes, 'flat') else [axes]

    for ax, img_path, lbl_path in zip(axes, images[:n_show], labels[:n_show]):
        img = Image.open(img_path)
        W, H = img.size
        parts = list(map(float, lbl_path.read_text().strip().split()))
        _, cx, cy, bw, bh = parts
        x1 = int((cx - bw/2) * W)
        y1 = int((cy - bh/2) * H)
        ax.imshow(img)
        rect = mpatches.Rectangle((x1,y1), int(bw*W), int(bh*H),
                                    linewidth=2, edgecolor="#5cb85c", facecolor="none")
        ax.add_patch(rect)
        ax.set_title(img_path.stem, color="#aab3bc", fontsize=8)
        ax.axis("off")

    for ax in list(axes)[n_show:]:
        ax.axis("off")

    fig.suptitle("Generated dataset — Waldo bbox (green)", color="#aab3bc", fontsize=12)
    plt.tight_layout()
    plt.show()

## 9 — YOLO Inference

`detect_waldo(image)` returns a list of `{bbox, confidence}` dicts sorted by confidence.  
**Requires a trained model** in `frontend/models/` or `frontend/data/models/`.  
If no model exists, the function returns an empty list (no crash).

In [ ]:
from backend.vision.inference.detect_waldo import detect_waldo, _default_model_path

model_path = _default_model_path()
print(f"Model path : {model_path}")
if model_path is None:
    print("⚠  No trained model found.")
    print("   Train first: python scripts/train_model.py")
    print("   Then re-run this section.")
else:
    print(f"✓  Model found: {model_path}")

In [ ]:
# Run inference on a freshly generated scene
test_scene, gt_bbox = generate_scene(difficulty="easy", seed=55)

detections = detect_waldo(test_scene)

if not detections:
    print("No detections (model not trained yet — this is expected before training).")
else:
    print(f"Detections: {len(detections)}")
    for i, det in enumerate(detections):
        print(f"  [{i}] bbox={det['bbox']}  conf={det['confidence']:.3f}")

In [ ]:
# Visualise inference result (works even with empty detections)
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(test_scene)
ax.axis("off")

# Ground truth (green)
x1, y1, x2, y2 = gt_bbox
gt_rect = mpatches.FancyBboxPatch(
    (x1, y1), x2-x1, y2-y1,
    boxstyle="round,pad=2",
    linewidth=3, edgecolor="#5cb85c", facecolor="none", label="Ground truth"
)
ax.add_patch(gt_rect)

# YOLO detections (blue, top-1 only)
if detections:
    dx1, dy1, dx2, dy2 = detections[0]["bbox"]
    conf = detections[0]["confidence"]
    yolo_rect = mpatches.FancyBboxPatch(
        (dx1, dy1), dx2-dx1, dy2-dy1,
        boxstyle="round,pad=2",
        linewidth=3, edgecolor="#5bc0de", facecolor="none",
        label=f"YOLO (conf={conf:.2f})"
    )
    ax.add_patch(yolo_rect)
    score = iou(gt_bbox, detections[0]["bbox"])
    ax.set_title(f"Inference — IoU={score:.3f}", color="#aab3bc", fontsize=12)
else:
    ax.set_title("Inference — no model (ground truth shown)", color="#aab3bc", fontsize=12)

ax.legend(loc="upper right", fontsize=9,
          facecolor="#3a3f44", labelcolor="#aab3bc", edgecolor="#484e55")
plt.tight_layout()
plt.show()

## 10 — Full Game Simulation

Simulate a complete game round end-to-end:  
generate scene → user guess → YOLO inference → compare → verdict.

In [ ]:
from backend.utils.bbox_utils import point_in_bbox

def simulate_game_round(
    difficulty: str = "medium",
    seed: int | None = None,
    user_click: tuple | None = None,  # (x, y) — None for random guess
) -> dict:
    """
    Simulate one full game round.

    Returns
    -------
    dict with keys: scene, gt_bbox, user_click, user_found,
                    detections, yolo_found, yolo_bbox, yolo_conf
    """
    rng = random.Random(seed)

    # 1. Generate scene
    scene, gt_bbox = generate_scene(difficulty=difficulty, seed=seed)
    W, H = scene.size

    # 2. User guess (random if not provided)
    if user_click is None:
        cx = rng.randint(0, W - 1)
        cy = rng.randint(0, H - 1)
        user_click = (cx, cy)

    user_found = point_in_bbox(user_click[0], user_click[1], gt_bbox)

    # 3. YOLO inference
    detections = detect_waldo(scene)
    yolo_bbox = detections[0]["bbox"] if detections else None
    yolo_conf = detections[0]["confidence"] if detections else None
    yolo_found = len(detections) > 0

    return dict(
        scene=scene, gt_bbox=gt_bbox,
        user_click=user_click, user_found=user_found,
        detections=detections, yolo_found=yolo_found,
        yolo_bbox=yolo_bbox, yolo_conf=yolo_conf,
    )


result = simulate_game_round(difficulty="medium", seed=42)

print("=" * 45)
print("GAME ROUND RESULT")
print("=" * 45)
print(f"Ground truth bbox : {result['gt_bbox']}")
print(f"User click        : {result['user_click']}")
print(f"User found Waldo  : {'✓ YES' if result['user_found'] else '✗ NO'}")
print(f"YOLO found Waldo  : {'✓ YES' if result['yolo_found'] else '✗ NO (no model)'}")
if result["yolo_conf"]:
    print(f"YOLO confidence   : {result['yolo_conf']:.3f}")

In [ ]:
# Visualise the full round
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(result["scene"])
ax.axis("off")

# Ground truth (green)
x1,y1,x2,y2 = result["gt_bbox"]
ax.add_patch(mpatches.FancyBboxPatch(
    (x1,y1), x2-x1, y2-y1, boxstyle="round,pad=2",
    linewidth=3, edgecolor="#5cb85c", facecolor="none", label="Ground truth"))

# User click (orange X)
ux, uy = result["user_click"]
ax.plot(ux, uy, "x", color="#f0ad4e", markersize=18, markeredgewidth=3.5,
        label=f"User click ({ux},{uy})")

# YOLO detection (blue)
if result["yolo_bbox"]:
    dx1,dy1,dx2,dy2 = result["yolo_bbox"]
    ax.add_patch(mpatches.FancyBboxPatch(
        (dx1,dy1), dx2-dx1, dy2-dy1, boxstyle="round,pad=2",
        linewidth=3, edgecolor="#5bc0de", facecolor="none",
        label=f"YOLO (conf={result['yolo_conf']:.2f})"))

# Verdict
u = result["user_found"]
y = result["yolo_found"]
if u and y:   verdict, vc = "Both found Waldo! 🏆", "#5cb85c"
elif u:       verdict, vc = "You beat the AI! 🥇", "#f0ad4e"
elif y:       verdict, vc = "AI wins! 🤖", "#5bc0de"
else:         verdict, vc = "Nobody found Waldo…", "#7a8288"

ax.set_title(verdict, color=vc, fontsize=14, fontweight="bold", pad=12)
ax.legend(loc="upper right", fontsize=9,
          facecolor="#3a3f44", labelcolor="#aab3bc", edgecolor="#484e55")
plt.tight_layout()
plt.show()

In [ ]:
# Batch simulation: run N rounds, track user (random) vs YOLO accuracy
N_ROUNDS = 20
user_wins, yolo_wins = 0, 0

for seed in range(N_ROUNDS):
    r = simulate_game_round(difficulty="medium", seed=seed)
    if r["user_found"]:  user_wins += 1
    if r["yolo_found"]: yolo_wins += 1

print(f"Batch simulation — {N_ROUNDS} rounds, difficulty=medium")
print(f"  Random user accuracy : {user_wins}/{N_ROUNDS}  ({user_wins/N_ROUNDS:.0%})")
print(f"  YOLO accuracy        : {yolo_wins}/{N_ROUNDS}  ({yolo_wins/N_ROUNDS:.0%})")
print()
print("Note: user accuracy is random clicking — expected ~(waldo_area / scene_area).")
print("      YOLO accuracy will be 0% until model is trained.")

## Quick Reference

```python
# Scene generation
from backend.scene_generation.generate_scene import generate_scene
scene, bbox = generate_scene(difficulty="medium", seed=42)

# YOLO inference
from backend.vision.inference.detect_waldo import detect_waldo
detections = detect_waldo(scene)   # list of {bbox, confidence}

# BBox helpers
from backend.utils.bbox_utils import point_in_bbox, iou, bbox_to_yolo, yolo_to_bbox
found = point_in_bbox(x, y, bbox)
score = iou(pred_bbox, gt_bbox)

# Dataset generation
from backend.vision.dataset.dataset_builder import generate_dataset
generate_dataset(n_images=2000)

# Project paths
from backend.utils.config import get_paths
paths = get_paths()
# paths.images_dir, paths.labels_dir, paths.models_dir, ...
```